In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

/home/bakar_qureshi/miniconda3/envs/ml-cuda12.6/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {DEVICE}")

if DEVICE == "cuda":
    
    device = torch.cuda.current_device()
    gpu_name = torch.cuda.get_device_name(device)

    total_memory = torch.cuda.get_device_properties(device).total_memory
    allocated_memory = torch.cuda.memory_allocated(device)
    reserved_memory = torch.cuda.memory_reserved(device)
    free_memory = total_memory - reserved_memory
    
    print(f"Backend: CUDA")
    print(f"GPU: {gpu_name}")
    print(f"Total Memory:     {total_memory / 1e6:.2f} MB | {total_memory / 1e9:.2f} GB")
    print(f"Allocated Memory: {allocated_memory / 1e6:.2f} MB | {allocated_memory / 1e9:.2f} GB")
    print(f"Reserved Memory:  {reserved_memory / 1e6:.2f} MB | {reserved_memory / 1e9:.2f} GB")
    print(f"Free Memory:      {free_memory / 1e6:.2f} MB | {free_memory / 1e9:.2f} GB")

elif DEVICE == "mps":
    # Note: MPS doesn't expose detailed memory stats like CUDA.
    # Apple Silicon uses unified memory (shared between CPU and GPU).
    # You can check total system memory as a proxy.
    import subprocess
    total_memory = int(subprocess.check_output(["sysctl", "-n", "hw.memsize"]).strip())
    
    print(f"Backend: MPS")
    print(f"Device: Apple Silicon (Metal Performance Shaders)")
    print(f"Total System Memory (unified): {total_memory / 1e6:.2f} MB | {total_memory / 1e9:.2f} GB")
    
    # Verify MPS works with a quick tensor test
    x = torch.tensor([1.0, 2.0]).to("mps")
    print(f"MPS tensor test: {x.device}")

else:
    print("No GPU available (no CUDA or MPS backend found)")

Using device: cuda
Backend: CUDA
GPU: NVIDIA GeForce GTX 1060
Total Memory:     6442.32 MB | 6.44 GB
Allocated Memory: 0.00 MB | 0.00 GB
Reserved Memory:  0.00 MB | 0.00 GB
Free Memory:      6442.32 MB | 6.44 GB


## Load Model

In [2]:
MODEL_NAME = "google/gemma-3-270m-it"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype="auto",
    device_map="auto",
    attn_implementation="eager"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Model using device: {model.device}")
print(f"Model dtype: {model.dtype}")

Loading weights: 100%|██████████| 236/236 [00:00<00:00, 256.42it/s]


Model using device: cuda:0
Model dtype: torch.bfloat16


In [4]:
tokenizer("hello my name is bakar")

{'input_ids': [2, 23391, 1041, 1463, 563, 143440], 'attention_mask': [1, 1, 1, 1, 1, 1]}

## Let's pass tokens to the model

In [5]:
input_string = "Hello my name is bakar"

token_tensor = torch.tensor(tokenizer(input_string)["input_ids"]).unsqueeze(0).to(model.device)

print(f"Token tensor shape: {token_tensor.shape}")
print(f"Token tensor device: {token_tensor.device}")
print(f"Token tensor dtype: {token_tensor.dtype}")
print(f"Token tensor: {token_tensor}")

outputs = model(token_tensor)
outputs.keys()

Token tensor shape: torch.Size([1, 6])
Token tensor device: cuda:0
Token tensor dtype: torch.int64
Token tensor: tensor([[     2,   9259,   1041,   1463,    563, 143440]], device='cuda:0')


odict_keys(['logits', 'past_key_values'])

In [6]:
logits = outputs.logits
logits.shape

torch.Size([1, 6, 262144])

In [7]:
predicted_ids = logits.argmax(dim=-1)
print(f"Predicted token IDs: {predicted_ids}")
print(f"Predicted token IDs shape: {predicted_ids.shape}")

Predicted token IDs: tensor([[  1106, 236888,   4389,    563,    870, 236761]], device='cuda:0')
Predicted token IDs shape: torch.Size([1, 6])


In [8]:
predicted_tokens = tokenizer.convert_ids_to_tokens(predicted_ids[0])
predicted_text = tokenizer.decode(predicted_ids[0])

print(f"Predicted tokens: {predicted_tokens}")
print(f"Predicted text: {predicted_text}")

print(f"Original input string: {input_string}")

Predicted tokens: ['import', '!', '▁friend', '▁is', '▁[', '.']
Predicted text: import! friend is [.
Original input string: Hello my name is bakar


In [12]:
def get_model_num_params(model):
    """
    Returns the number of trainable, non-trainable and total parameters of a PyTorch model.
    """
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    non_trainable_params = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    total_params = trainable_params + non_trainable_params
    return {"trainable_params": trainable_params,
            "non_trainable_params": non_trainable_params,
            "total_params": total_params}

# Get parameters of our fine-tuned model
model_params = get_model_num_params(model)
print(f"Trainable parameters: {model_params['trainable_params']:,}")
print(f"Non-trainable parameters: {model_params['non_trainable_params']:,}")
print(f"Total parameters: {model_params['total_params']:,}")

Trainable parameters: 268,098,176
Non-trainable parameters: 0
Total parameters: 268,098,176


In [2]:
import torch

In [3]:
torch.cuda.empty_cache()

In [4]:
def functionality(content: str) -> int:

    return 1

In [5]:
functionality("test")

1

In [ ]:
import torch

def get_gpu_info():

    if torch.cuda.is_available():
    
        device_idx = torch.cuda.current_device()
        gpu_name = torch.cuda.get_device_name(device_idx)    
        total_vram = torch.cuda.get_device_properties(device_idx).total_memory / (1024 ** 3)
    

    return (gpu_name, total_vram)